In [ ]:
import pandas as pd
import numpy as np
import torch 

df = pd.read_csv("data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["player" , "date"]).reset_index(drop = True)

print(df.shape)

In [ ]:
sequence_features = ["runs", "balls_faced", "fours", "sixes", "strike_rate",
    "wickets", "runs_conceded", "balls_bowled", "economy", 
    "maidens", "total_fantasy_points"]

context_features = [ "venue_avg_fantasy", "venue_std_fantasy",
    "opposition_avg_fantasy", "opposition_std_fantasy",
    "venue_first_appearance", "opposition_first_appearance",
    "is_home", "won_toss", "role_encoded", "batting_position",
    "matches_played", "expanding_season_fantasy_avg",
    "Total_career_runs", "Total_career_wickets"]

SEQ_LEN = 7

X_seq = []
X_context = []
y = []

for player , group in df.groupby(["player"]):
    player_data = group[sequence_features].values
    context_data = group[context_features].values
    targets = group["total_fantasy_points"].values

    for i in range(len(player_data)):

        if i < SEQ_LEN:
            pad_len = SEQ_LEN - i
            real_data = player_data[:i]
            padding = np.zeros((pad_len, len(sequence_features)))
            seq = np.vstack([padding,real_data])
        
        else:
            seq = player_data[i-SEQ_LEN:i]
        
        X_seq.append(seq)
        X_context.append(context_data[i])
        y.append(targets[i])


X_seq = np.array(X_seq)
X_context = np.array(X_context)
y = np.array(y)

print(X_seq.shape)      # should be (27909, 7, 5)
print(X_context.shape)  # should be (27909, 8)
print(y.shape)          # should be (27909,)

In [ ]:
split_idx = df[df["date"].dt.year >= 2025].index[0]
print(split_idx)

In [ ]:
df_sorted = df.sort_values("date").reset_index(drop = True)

In [ ]:
split_idx = df_sorted[df_sorted["date"].dt.year >= 2025].index[0]
print(split_idx)

In [ ]:
split = 24367
X_seq_train = X_seq[:split]
X_seq_test = X_seq[split:]

X_context_train = X_context[:split]
X_context_test = X_context[split:]

y_train = y[:split]
y_test = y[split:]

print(X_seq_train.shape)
print(X_seq_test.shape)
print(X_context_train.shape)

In [ ]:
X_seq_train_t = torch.FloatTensor(X_seq_train)
X_seq_test_t = torch.FloatTensor(X_seq_test)

X_context_train_t = torch.FloatTensor(X_context_train)
X_context_test_t = torch.FloatTensor(X_context_test)

y_train_t = torch.FloatTensor(y_train)
y_test_t = torch.FloatTensor(y_test)

print(X_seq_train_t.shape)
print(X_context_train_t.shape)


In [ ]:
from torch.utils.data import TensorDataset , DataLoader

train_dataset = TensorDataset(X_seq_train_t,X_context_train_t,y_train_t)
test_dataset = TensorDataset(X_seq_test_t,X_context_test_t,y_test_t)

train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size=32,shuffle = False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
import torch.nn as nn
torch.manual_seed(42)
torch.mps.manual_seed(42)
import numpy as np
np.random.seed(42)

class CricketLSTM(nn.Module):
    def __init__(self,seq_features,context_features,hidden_size = 32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size = seq_features,
            hidden_size = hidden_size,
            num_layers = 2,
            batch_first = True,
            dropout = 0.3
        )

        self.fc1 = nn.Linear(hidden_size + context_features ,32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32,1)

    def forward(self,seq,context):

        lstm_out , (hidden,cell) = self.lstm(seq)
        last_output = lstm_out[:,-1,:]
        combined = torch.cat([last_output,context],dim = 1)

        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x.squeeze(1)






In [ ]:
model = CricketLSTM(
    seq_features = 11,
    context_features = 14,
    hidden_size = 32
)

print(model)

In [ ]:
device = torch.device("mps")
model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.0005
)

print(f"Using Device : {device}")


In [ ]:
print(next(model.parameters()).device)

In [ ]:
EPOCHS = 200
best_mae = float('inf')
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for seq,context,target in train_loader:

        seq = seq.to(device)
        context = context.to(device)
        target = target.to(device)

        #forward pass
        optimizer.zero_grad()
        predcitions = model(seq,context)
        loss = criterion(predcitions,target)

        #backwardpass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_preds = []
            test_targets = []
            for seq, context, target in test_loader:
                seq = seq.to(device)
                context = context.to(device)
                pred = model(seq, context)
                test_preds.extend(pred.cpu().numpy())
                test_targets.extend(target.numpy())
        
        test_mae = np.mean(np.abs(np.array(test_preds) - np.array(test_targets)))
        if test_mae < best_mae:
            best_mae = test_mae
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "models/best_lstm.pt")
            print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss/len(train_loader):.2f}, Test MAE: {test_mae:.2f} ← best saved")
        else:
            print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss/len(train_loader):.2f}, Test MAE: {test_mae:.2f}")

print(f"\nBest MAE: {best_mae:.2f} at epoch {best_epoch}")

        

In [ ]:
print(torch.isnan(X_seq_train_t).any())
print(torch.isnan(X_context_train_t).any())
print(torch.isnan(y_train_t).any())

In [ ]:
context_features = ["rolling_avg_fantasy_5", "venue_avg_fantasy", "opposition_avg_fantasy",
                    "is_home", "won_toss", "role_encoded", "batting_position", "matches_played"]

print(df[context_features].isnull().sum())

In [ ]:
df.columns

In [ ]:
print(len(sequence_features))
print(len(context_features))